# Conference-Style Results Summary

This notebook summarizes `testscript.py` outputs into compact paper-ready artifacts:
1. Main summary table (overall by agent)
2. Scenario robustness heatmap (composite rank)
3. Pareto tradeoff scatter (allocation vs handovers)
4. Critical-case time series (hard scenarios only)

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE_DIR = Path('.').resolve()
AGENTS = ['BASELINE', 'PPO', 'DQN', 'ODT', 'ODT_FINETUNED', 'ORACLE']
SCENARIOS = [
    'no_scenario',
    'load_cycle_1',
    'load_cycle_2',
    'load_cycle_5',
    'medium_aircraft',
    'large_aircraft',
    'snr_congested',
]

LATENCY_CLIP_MS = 1000  # for optional filtered latency displays

print('Base dir:', BASE_DIR)

Base dir: /Users/hindmukhtar/Documents/GitHub/DLRL2025/Single Constellation 


In [2]:
def load_runs(base_dir: Path, agents, scenarios):
    rows = []
    missing = []

    # Ensure we point to the folder containing the CSVs
    base_dir = Path(base_dir)
    if not (base_dir / "BASELINE_observations_no_scenario.csv").exists():
        candidate = base_dir / "Single Constellation "
        if candidate.exists():
            base_dir = candidate

    for agent in agents:
        for scenario in scenarios:
            # Try expected file first
            candidates = [
                base_dir / f"{agent}_observations_{scenario}.csv",
            ]


            fp = next((p for p in candidates if p.exists()), None)
            if fp is None:
                missing.append(f"{agent}_observations_{scenario}.csv")
                continue

            df = pd.read_csv(fp)

            # Normalize headers from CSVs written with spaces after commas
            df.columns = (
                df.columns
                .str.strip()
                .str.replace(r"\s+", "_", regex=True)
            )
            df["agent"] = agent
            df["scenario"] = scenario

            rows.append(df)

    data = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return data, missing, base_dir

data, missing, used_dir = load_runs(BASE_DIR, AGENTS, SCENARIOS)
print("Using dir:", used_dir)
print("Loaded rows:", len(data))
print("Loaded agent/scenario pairs:",
      data[["agent", "scenario"]].drop_duplicates().shape[0] if len(data) else 0)
if missing:
    print("Missing files (if any):", ", ".join(missing[:12]), "..." if len(missing) > 12 else "")
data.head()


Using dir: /Users/hindmukhtar/Documents/GitHub/DLRL2025/Single Constellation 
Loaded rows: 45444
Loaded agent/scenario pairs: 42


,step,lat,lon,alt,snr,load,handovers,allocated_bw,allocation_ratio,demand_MB,...,queing_delay_s,propagation_latency_s,transmission_rate_mbps,latency_req_s,beam_capacity,service_drop_s,dwell_remaining_s,ttt_remaining_s,agent,scenario
0,0,29.956900,-95.336899,327.660004,18.503132,0.659110,0.0,0.000000,0.0,0.562500,...,1000.0,0.052708,0.0,0.2,60.000000,1.0,5.0,0.0,BASELINE,no_scenario
1,1,29.954182,-95.333305,365.760010,8.830511,0.658336,0.0,3.070534,1.0,3.070534,...,0.0,0.053989,4.5,0.2,327.523621,0.0,4.0,0.0,BASELINE,no_scenario
2,2,29.951462,-95.329712,403.859985,8.818870,0.667468,0.0,2.986333,1.0,2.986333,...,0.0,0.053196,4.5,0.2,318.542236,0.0,0.0,5.0,BASELINE,no_scenario
3,3,29.948744,-95.326118,441.959991,17.458019,0.678836,1.0,2.906630,1.0,2.906630,...,0.0,0.052915,4.5,0.2,310.040527,0.0,5.0,0.0,BASELINE,no_scenario
4,4,29.945999,-95.322273,472.440002,9.066678,0.664004,1.0,2.934174,1.0,2.934174,...,0.0,0.052500,4.5,0.2,312.978577,0.0,0.0,5.0,BASELINE,no_scenario


In [3]:
data['latency_s'] = data['queing_delay_s'] + data['propagation_latency_s']

In [9]:
def episode_metrics(df):
    out = []
    for (agent, scenario), g in df.groupby(['agent', 'scenario']):
        alloc_ratio = float(g['allocation_ratio'].mean())
        lat_req = g['latency_req_s'] if 'latency_req_s' in g.columns else pd.Series([np.inf] * len(g))
        lat_violation_rate = float((g['latency_s'] > lat_req).mean())
        service_drop_s = float(g['service_drop_s'].sum()) if 'service_drop_s' in g.columns else 0.0
        total_handovers = float(g['handovers'].max()) if 'handovers' in g.columns else np.nan
        avg_latency_ms = float(g['latency_s'].mean() * 1000.0)

        # Composite score: higher is better.
        J = (
            1.0 * alloc_ratio
            # - 0.01 * lat_violation_rate
            - 0.01 * service_drop_s
        )

        out.append({
            'agent': agent,
            'scenario': scenario,
            'allocation_ratio': alloc_ratio,
            'latency_violation_rate': lat_violation_rate,
            'service_drop_s': service_drop_s,
            'total_handovers': total_handovers,
            'avg_latency_ms': avg_latency_ms,
            'composite_score': J,
        })
    return pd.DataFrame(out)

m = episode_metrics(data)
m.sort_values(['scenario','composite_score'], ascending=[True, False]).head(20)

,agent,scenario,allocation_ratio,latency_violation_rate,service_drop_s,total_handovers,avg_latency_ms,composite_score
35,PPO,large_aircraft,0.999397,0.072089,0.000000,231.0,59.754849,0.999397
28,ORACLE,large_aircraft,0.997035,0.080407,0.000000,255.0,79.284733,0.997035
7,DQN,large_aircraft,0.995590,0.075786,0.000000,248.0,384.047341,0.995590
14,ODT,large_aircraft,0.979570,0.095194,74.010345,256.0,14270.073900,0.239467
21,ODT_FINETUNED,large_aircraft,0.979570,0.095194,74.010345,256.0,14270.073900,0.239467
0,BASELINE,large_aircraft,0.977246,0.097043,75.104881,257.0,15128.646799,0.226197
36,PPO,load_cycle_1,1.000000,0.007394,0.000000,230.0,54.783544,1.000000
8,DQN,load_cycle_1,0.998405,0.009242,0.000000,250.0,112.833105,0.998405
29,ORACLE,load_cycle_1,0.993784,0.013863,24.688061,256.0,4733.882076,0.746904
15,ODT,load_cycle_1,0.983673,0.025878,74.010345,255.0,13982.948817,0.243570


## 1) Main Summary Table (Overall by Agent)

In [10]:
overall = (
    m.groupby('agent', as_index=False)
     .agg({
         'allocation_ratio': 'mean',
         'latency_violation_rate': 'mean',
         'service_drop_s': 'mean',
         'total_handovers': 'mean',
         'avg_latency_ms': 'mean',
         'composite_score': 'mean',
     })
)
overall = overall.sort_values('composite_score', ascending=False)
overall.style.format({
    'allocation_ratio': '{:.4f}',
    'latency_violation_rate': '{:.2%}',
    'service_drop_s': '{:.2f}',
    'total_handovers': '{:.1f}',
    'avg_latency_ms': '{:.2f}',
    'composite_score': '{:.4f}',
})

,agent,allocation_ratio,latency_violation_rate,service_drop_s,total_handovers,avg_latency_ms,composite_score
1,DQN,0.9926,2.96%,30.68,247.6,5833.34,0.6858
4,ORACLE,0.9916,3.21%,35.07,255.3,6591.68,0.6409
5,PPO,0.9909,3.20%,42.85,241.4,8024.84,0.5624
2,ODT,0.9824,4.15%,78.75,255.6,14833.15,0.1949
3,ODT_FINETUNED,0.9824,4.15%,78.75,255.6,14833.15,0.1949
0,BASELINE,0.9693,5.45%,145.39,259.1,27766.74,-0.4846


## 2) Scenario Robustness Heatmap (Composite Rank)

In [11]:
rank_df = m.copy()
rank_df['rank_in_scenario'] = rank_df.groupby('scenario')['composite_score'].rank(ascending=False, method='min')
pivot_rank = rank_df.pivot(index='scenario', columns='agent', values='rank_in_scenario')
pivot_rank = pivot_rank.reindex(index=SCENARIOS, columns=[a for a in AGENTS if a in pivot_rank.columns])

fig = px.imshow(
    pivot_rank,
    text_auto='.0f',
    aspect='auto',
    color_continuous_scale='RdYlGn_r',
    title='Scenario Robustness (Rank 1 = Best)'
)
fig.update_layout(height=420)
fig.show()

## 4) Critical-Case Time Series (Hard Scenarios Only)

In [7]:
HARD_SCENARIOS = ['snr_congested', 'large_aircraft']
plot_df = data[data['scenario'].isin(HARD_SCENARIOS)].copy()

if len(plot_df) == 0:
    print('No hard-scenario data found.')
else:
    # Smooth for paper readability while preserving trend.
    plot_df = plot_df.sort_values(['scenario', 'agent', 'step'])
    plot_df['alloc_ratio_smooth'] = (
        plot_df.groupby(['scenario', 'agent'])['allocation_ratio']
               .transform(lambda s: s.rolling(15, min_periods=1).mean())
    )
    plot_df['service_drop_cum'] = plot_df.groupby(['scenario', 'agent'])['service_drop_s'].cumsum()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                        subplot_titles=['Allocation Ratio (smoothed)', 'Cumulative Service Drop (s)'])

    for scenario in HARD_SCENARIOS:
        for agent in sorted(plot_df['agent'].unique()):
            g = plot_df[(plot_df['scenario'] == scenario) & (plot_df['agent'] == agent)]
            if g.empty:
                continue
            name = f'{agent} | {scenario}'
            fig.add_trace(
                go.Scatter(x=g['step'], y=g['alloc_ratio_smooth'], mode='lines', name=name, legendgroup=name),
                row=1, col=1
            )
            fig.add_trace(
                go.Scatter(x=g['step'], y=g['service_drop_cum'], mode='lines', name=name, legendgroup=name, showlegend=False),
                row=2, col=1
            )

    fig.update_yaxes(title_text='Allocation/Demand', row=1, col=1)
    fig.update_yaxes(title_text='Seconds', row=2, col=1)
    fig.update_xaxes(title_text='Step', row=1, col=1)
    fig.update_xaxes(title_text='Step', row=2, col=1)
    fig.update_layout(height=780, title='Critical-Case Temporal Behavior', legend_title='Agent | Scenario')
    fig.show()

## Optional: Metric-by-Scenario Table (Paper Appendix)

In [8]:
table = m.copy()
table['agent_scenario'] = table['agent'] + ' | ' + table['scenario']
metric_table = table.set_index('agent_scenario')[['allocation_ratio', 'latency_violation_rate', 'service_drop_s', 'total_handovers', 'avg_latency_ms', 'composite_score']]
metric_table.sort_values('composite_score', ascending=False).head(30)

,allocation_ratio,latency_violation_rate,service_drop_s,total_handovers,avg_latency_ms,composite_score
agent_scenario,,,,,,
DQN | no_scenario,1.000000,0.007394,0.000000,246.0,54.755908,1.000000
PPO | no_scenario,1.000000,0.007394,0.000000,230.0,54.769756,1.000000
PPO | load_cycle_1,1.000000,0.007394,0.000000,230.0,54.783544,1.000000
PPO | large_aircraft,0.999397,0.072089,0.000000,231.0,59.754849,0.999397
DQN | load_cycle_1,0.998405,0.009242,0.000000,250.0,112.833105,0.998405
ORACLE | load_cycle_2,0.998071,0.010166,9.569111,261.0,1903.714691,0.998071
PPO | load_cycle_2,0.998071,0.010166,9.569111,237.0,1903.708166,0.998071
ODT_FINETUNED | load_cycle_2,0.998071,0.010166,9.569111,253.0,1903.620157,0.998071
DQN | load_cycle_2,0.998071,0.010166,9.569111,249.0,1903.720386,0.998071
